<a href="https://colab.research.google.com/github/Ihsan-Fazal/LULC/blob/main/SegFormer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q transformers

In [3]:
import numpy as np
import torch
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from transformers import SegformerForSemanticSegmentation
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    cohen_kappa_score,
    jaccard_score,
    classification_report
)

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

PyTorch version: 2.11.0+cu128
GPU available: True
Device: cuda


In [4]:
X_train = np.load('/content/drive/MyDrive/X_train.npy')
Y_train_mask = np.load('/content/drive/MyDrive/Y_train_mask.npy')

X_test = np.load('/content/drive/MyDrive/X_test.npy')
Y_test_mask = np.load('/content/drive/MyDrive/Y_test_mask.npy')

print("X_train:", X_train.shape)
print("Y_train_mask:", Y_train_mask.shape)
print("X_test:", X_test.shape)
print("Y_test_mask:", Y_test_mask.shape)

X_train: (639, 256, 256, 7)
Y_train_mask: (639, 256, 256)
X_test: (125, 256, 256, 7)
Y_test_mask: (125, 256, 256)


In [5]:
class_map = {
    10: 0,   # Tree cover
    20: 1,   # Shrubland
    30: 2,   # Grassland
    40: 3,   # Cropland
    60: 3,   # Bare / Sparse vegetation -> Cropland
    50: 4,   # Built-up
    80: 5,   # Permanent water
    90: 5    # Herbaceous wetland -> Permanent water
}

lut = np.full(101, -1, dtype=np.int64)

for old_id, new_id in class_map.items():
    lut[old_id] = new_id

Y_train_ready = lut[Y_train_mask]
Y_test_ready = lut[Y_test_mask]

print("Unique training classes:", np.unique(Y_train_ready))
print("Unique testing classes:", np.unique(Y_test_ready))

Unique training classes: [0 1 2 3 4 5]
Unique testing classes: [0 1 2 3 4 5]


In [6]:
#Convert Data to pytorch format
X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)

X_train = np.transpose(X_train, (0, 3, 1, 2))
X_test = np.transpose(X_test, (0, 3, 1, 2))

Y_train_ready = Y_train_ready.astype(np.int64)
Y_test_ready = Y_test_ready.astype(np.int64)

print("X_train:", X_train.shape)
print("Y_train:", Y_train_ready.shape)
print("X_test:", X_test.shape)
print("Y_test:", Y_test_ready.shape)

X_train: (639, 7, 256, 256)
Y_train: (639, 256, 256)
X_test: (125, 7, 256, 256)
Y_test: (125, 256, 256)


In [7]:
class LULCDataset(Dataset):
    def __init__(self, images, masks):
        self.images = torch.tensor(images, dtype=torch.float32)
        self.masks = torch.tensor(masks, dtype=torch.long)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return {
            "pixel_values": self.images[idx],
            "labels": self.masks[idx]
        }


train_dataset = LULCDataset(X_train, Y_train_ready)
test_dataset = LULCDataset(X_test, Y_test_ready)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Training batches:", len(train_loader))
print("Testing batches:", len(test_loader))

Training batches: 160
Testing batches: 32


In [8]:
model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/mit-b0",
    num_labels=6,
    num_channels=7,
    ignore_mismatched_sizes=True
)

model = model.to(device)

print("SegFormer-B0 loaded.")

config.json:   0%|          | 0.00/70.0k [00:00<?, ?B/s]

[transformers] You passed `num_labels=6` which is incompatible to the `id2label` map of length `1000`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 14.4MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b0
Key                                                     | Status     |                                                                                                 
--------------------------------------------------------+------------+-------------------------------------------------------------------------------------------------
classifier.bias                                         | UNEXPECTED |                                                                                                 
classifier.weight                                       | UNEXPECTED |                                                                                                 
decode_head.batch_norm.num_batches_tracked              | MISSING    |                                                                                                 
decode_head.linear_projections.{0, 1, 2, 3}.proj.weight | MISSING    |          

model.safetensors: reconstructing file:   0%|          |  0.00B / 14.3MB            

model.safetensors: downloading bytes:           |  0.00B            

SegFormer-B0 loaded.


In [9]:
class_counts = np.bincount(
    Y_train_ready.flatten(),
    minlength=6
)

print("Class pixel counts:")
for i, count in enumerate(class_counts):
    print(i, count)

Class pixel counts:
0 6236627
1 6704273
2 9238845
3 7137836
4 5275288
5 7284635


In [10]:
class_weights = 1.0 / np.sqrt(class_counts)

class_weights = class_weights / class_weights.mean()

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(device)

print("Class weights:", class_weights)

Class weights: tensor([1.0465, 1.0093, 0.8598, 0.9782, 1.1379, 0.9683], device='cuda:0')


In [11]:
criterion = torch.nn.CrossEntropyLoss(
    weight=class_weights
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

In [12]:
num_epochs = 15

train_losses = []

for epoch in range(num_epochs):

    model.train()
    running_loss = 0.0

    for batch in train_loader:

        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            pixel_values=pixel_values
        )

        # SegFormer outputs lower-resolution logits.
        # Resize them to the original 256x256 mask size.
        logits = F.interpolate(
            outputs.logits,
            size=labels.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

        loss = criterion(logits, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)
    train_losses.append(epoch_loss)

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Loss: {epoch_loss:.4f}"
    )

Epoch [1/15] Loss: 1.1383
Epoch [2/15] Loss: 0.8722
Epoch [3/15] Loss: 0.7923
Epoch [4/15] Loss: 0.7499
Epoch [5/15] Loss: 0.6909
Epoch [6/15] Loss: 0.6791
Epoch [7/15] Loss: 0.6564
Epoch [8/15] Loss: 0.6430
Epoch [9/15] Loss: 0.6087
Epoch [10/15] Loss: 0.6070
Epoch [11/15] Loss: 0.5906
Epoch [12/15] Loss: 0.5715
Epoch [13/15] Loss: 0.5643
Epoch [14/15] Loss: 0.5638
Epoch [15/15] Loss: 0.5628


In [13]:
model.eval()

all_predictions = []

with torch.no_grad():

    for batch in test_loader:

        pixel_values = batch["pixel_values"].to(device)

        outputs = model(
            pixel_values=pixel_values
        )

        logits = F.interpolate(
            outputs.logits,
            size=(256, 256),
            mode="bilinear",
            align_corners=False
        )

        predictions = torch.argmax(
            logits,
            dim=1
        )

        all_predictions.append(
            predictions.cpu().numpy()
        )

y_pred_segformer = np.concatenate(
    all_predictions,
    axis=0
)

print("Prediction shape:", y_pred_segformer.shape)

Prediction shape: (125, 256, 256)


In [14]:
def pixels_to_patch_label(y_array):
    patch_labels = []

    for i in range(y_array.shape[0]):

        pixels = y_array[i].flatten()

        counts = np.bincount(
            pixels,
            minlength=6
        )

        majority_class = np.argmax(counts)

        patch_labels.append(majority_class)

    return np.array(patch_labels)

In [15]:
y_test_patch_true = pixels_to_patch_label(Y_test_ready)

y_test_patch_pred_segformer = pixels_to_patch_label(
    y_pred_segformer
)

print("True patch labels:", y_test_patch_true.shape)
print("Predicted patch labels:", y_test_patch_pred_segformer.shape)

True patch labels: (125,)
Predicted patch labels: (125,)


In [16]:
patch_acc = accuracy_score(
    y_test_patch_true,
    y_test_patch_pred_segformer
)

patch_f1 = f1_score(
    y_test_patch_true,
    y_test_patch_pred_segformer,
    average="macro"
)

patch_kappa = cohen_kappa_score(
    y_test_patch_true,
    y_test_patch_pred_segformer
)

patch_iou = jaccard_score(
    y_test_patch_true,
    y_test_patch_pred_segformer,
    average="macro"
)

print("\n--- SEGFORMER-B0 RESULTS ---")
print(f"Test Accuracy (Patch): {patch_acc:.4f}")
print(f"Test Macro F1 (Patch): {patch_f1:.4f}")
print(f"Cohen's Kappa:         {patch_kappa:.4f}")
print(f"Mean IoU:              {patch_iou:.4f}")

target_names = [
    "Tree cover",
    "Shrubland",
    "Grassland",
    "Cropland",
    "Built-up",
    "Permanent water"
]

print(
    classification_report(
        y_test_patch_true,
        y_test_patch_pred_segformer,
        labels=np.arange(6),
        target_names=target_names,
        digits=4,
        zero_division=0
    )
)


--- SEGFORMER-B0 RESULTS ---
Test Accuracy (Patch): 0.8720
Test Macro F1 (Patch): 0.8671
Cohen's Kappa:         0.8409
Mean IoU:              0.7759
                 precision    recall  f1-score   support

     Tree cover     1.0000    0.8333    0.9091         6
      Shrubland     0.7143    0.7692    0.7407        13
      Grassland     0.7333    0.7857    0.7586        28
       Cropland     0.9630    0.8966    0.9286        29
       Built-up     0.9048    0.8636    0.8837        22
Permanent water     0.9643    1.0000    0.9818        27

       accuracy                         0.8720       125
      macro avg     0.8799    0.8581    0.8671       125
   weighted avg     0.8775    0.8720    0.8736       125

